# Q4 — Day-to-Day Price Volatility: DDR5 vs. NVMe

**Research question:** Within the same product and ZIP market, are DDR5 prices more volatile day to day than NVMe prices?

This analysis uses the standard deviation and coefficient of variation (CV) of daily median listing prices. A cohort is one `(series_id, geography_id, product_name)` combination with at least three observed dates. This controls for product mix and ZIP-market differences rather than comparing unrelated listings. The 12 ZIP markets are reported separately as a robustness check.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.matplotlib'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
import seaborn as sns

from src.cleaning import add_flags
from src.io import load_raw

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

df = add_flags(load_raw(validate=True))
df = df[~df['is_noise']].copy()
df['price_amount'] = pd.to_numeric(df['price_amount'], errors='coerce')
df = df.dropna(subset=['price_amount', 'observed_date', 'product_name', 'geography_id'])
print(f'Analysis rows after removing flagged system/NAS noise: {len(df):,}')

## Cohort construction

There can be more than one observation for a product, ZIP, and date. We first take that day's median, then calculate volatility across dates. Cohorts with fewer than three dates are excluded because a one- or two-point standard deviation is not a useful day-to-day stability measure.

In [ ]:
daily = (df.groupby(
    ['series_id', 'geography_id', 'product_name', 'observed_date'],
    as_index=False, observed=True
)['price_amount'].median().rename(columns={'price_amount': 'daily_median_price'}))

cohorts = (daily.groupby(
    ['series_id', 'geography_id', 'product_name'], as_index=False, observed=True
).agg(
    n_days=('observed_date', 'nunique'),
    mean_price=('daily_median_price', 'mean'),
    sd_price=('daily_median_price', 'std'),
))
cohorts = cohorts[cohorts['n_days'] >= 3].copy()
cohorts['cv_pct'] = 100 * cohorts['sd_price'] / cohorts['mean_price']

summary = (cohorts.groupby('series_id', as_index=False, observed=True)
               .agg(cohorts=('product_name', 'size'),
                    median_sd_usd=('sd_price', 'median'),
                    mean_sd_usd=('sd_price', 'mean'),
                    median_cv_pct=('cv_pct', 'median'),
                    mean_cv_pct=('cv_pct', 'mean'),
                    median_days=('n_days', 'median')))
summary['category'] = summary['series_id'].map({'ddr5_memory': 'DDR5', 'nvme_ssd': 'NVMe'})
display(summary[['category', 'cohorts', 'median_sd_usd', 'mean_sd_usd', 'median_cv_pct', 'mean_cv_pct', 'median_days']])

## Statistical comparison and ZIP-market robustness

The Mann–Whitney U test tests whether DDR5 cohort volatility is greater than NVMe cohort volatility. This is a distributional comparison, not a causal model. ZIP summaries show whether the direction is consistent across the 12 markets.

In [ ]:
ddr5 = cohorts[cohorts['series_id'] == 'ddr5_memory']
nvme = cohorts[cohorts['series_id'] == 'nvme_ssd']
test_sd = mannwhitneyu(ddr5['sd_price'], nvme['sd_price'], alternative='greater')
test_cv = mannwhitneyu(ddr5['cv_pct'], nvme['cv_pct'], alternative='greater')
tests = pd.DataFrame([
    {'measure': 'Standard deviation (USD)', 'DDR5 median': ddr5['sd_price'].median(), 'NVMe median': nvme['sd_price'].median(), 'U': test_sd.statistic, 'p_value': test_sd.pvalue},
    {'measure': 'Coefficient of variation (%)', 'DDR5 median': ddr5['cv_pct'].median(), 'NVMe median': nvme['cv_pct'].median(), 'U': test_cv.statistic, 'p_value': test_cv.pvalue},
])
display(tests)

zip_summary = (cohorts.groupby(['series_id', 'geography_id'], as_index=False, observed=True)
                   .agg(cohorts=('product_name', 'size'),
                        median_sd_usd=('sd_price', 'median'),
                        median_cv_pct=('cv_pct', 'median')))
zip_summary['category'] = zip_summary['series_id'].map({'ddr5_memory': 'DDR5', 'nvme_ssd': 'NVMe'})
display(zip_summary[['geography_id', 'category', 'cohorts', 'median_sd_usd', 'median_cv_pct']]
        .sort_values(['geography_id', 'category']))

In [ ]:
plot_data = cohorts.copy()
plot_data['category'] = plot_data['series_id'].map({'ddr5_memory': 'DDR5', 'nvme_ssd': 'NVMe'})
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=plot_data, x='category', y='sd_price', ax=axes[0], hue='category', legend=False)
axes[0].set(title='Cohort price volatility', xlabel='', ylabel='Standard deviation (USD)')
sns.boxplot(data=plot_data, x='category', y='cv_pct', ax=axes[1], hue='category', legend=False)
axes[1].set(title='Relative price volatility', xlabel='', ylabel='Coefficient of variation (%)')
plt.tight_layout()
plt.show()

print('Conclusion: DDR5 has materially higher typical within-product, within-ZIP volatility than NVMe in this snapshot; report both absolute and relative measures because NVMe prices are often unchanged.')

## Limitations

- The snapshot covers 29 dates, so this measures short-run listing-price stability rather than long-run market volatility.
- Repeated scraper observations are reduced to a daily median, but the analysis does not model serial correlation.
- The Mann–Whitney test treats product/ZIP cohorts as independent; results should be interpreted descriptively.
- A zero standard deviation means the observed listing price did not change in the available dates, not that the product had no unobserved market variation.